# Advence Rag with HAsh index

# Load the Pdfs


In [59]:
from langchain_community.document_loaders import DirectoryLoader , UnstructuredWordDocumentLoader

loader=DirectoryLoader(
    "docs",
    glob="*.docx",
    loader_cls=UnstructuredWordDocumentLoader
)

documents=loader.load()

# Split into Chunks

In [60]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks=splitter.split_documents(documents)

# Create A Hash

In [61]:
import hashlib

def get_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

In [39]:
text = "Machine Learning is fun"

print(get_hash(text))

36720634952eae5b19192816a68c77f320bf1d1e06149c72559107caa39c3582


# Attach Hash to metadata

In [62]:
for chunk in chunks:
    chunk.metadata["hash"]=get_hash(chunk.page_content)


print(chunk.metadata)

{'source': 'docs\\Renewable_Energy_and_Sustainable_Future.docx', 'hash': 'e359286d6af1b590f0a2e5bfe8b1c56f2f5516d7e713c5c1f106ee8fe85e3afc'}


In [42]:
for key, value in chunks[0].metadata.items():
    print(key, ":", value)

source : docs\Artificial_Intelligence_in_Healthcare.docx
hash : 212936a16e976b2325ddf0d1fa44c03c55a5a78834055447591b02befd1cc752


# Generate IDS

In [63]:
ids=[]

for i , chunk in enumerate(chunks):
    filename=chunk.metadata['source'].split("\\")[-1]
   

    chunk_id=f"{filename}_chunks{i}"

    ids.append(chunk_id)


print(ids[0])

Artificial_Intelligence_in_Healthcare.docx_chunks0


# Create Chroma Db

In [45]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db=Chroma(
    collection_name="word_books",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Insert

In [46]:
db.add_documents(

    documents=chunks,
    ids=ids
)

['Artificial_Intelligence_in_Healthcare.docx_chunks0',
 'Artificial_Intelligence_in_Healthcare.docx_chunks1',
 'Artificial_Intelligence_in_Healthcare.docx_chunks2',
 'Artificial_Intelligence_in_Healthcare.docx_chunks3',
 'Artificial_Intelligence_in_Healthcare.docx_chunks4',
 'Artificial_Intelligence_in_Healthcare.docx_chunks5',
 'Artificial_Intelligence_in_Healthcare.docx_chunks6',
 'Artificial_Intelligence_in_Healthcare.docx_chunks7',
 'Artificial_Intelligence_in_Healthcare.docx_chunks8',
 'Artificial_Intelligence_in_Healthcare.docx_chunks9',
 'Artificial_Intelligence_in_Healthcare.docx_chunks10',
 'Artificial_Intelligence_in_Healthcare.docx_chunks11',
 'Cybersecurity_in_the_Digital_Age.docx_chunks12',
 'Cybersecurity_in_the_Digital_Age.docx_chunks13',
 'Cybersecurity_in_the_Digital_Age.docx_chunks14',
 'Cybersecurity_in_the_Digital_Age.docx_chunks15',
 'Cybersecurity_in_the_Digital_Age.docx_chunks16',
 'Cybersecurity_in_the_Digital_Age.docx_chunks17',
 'Cybersecurity_in_the_Digital_A

In [64]:
existing = db.get(include=["metadatas"])
existing_lookup = {}

for id_, metadata in zip(existing["ids"], existing["metadatas"]):
    existing_lookup[id_] = metadata["hash"]

In [67]:
new_chunks = []
new_ids = []

for chunk, chunk_id in zip(chunks, ids):

    new_hash = chunk.metadata["hash"]

    if chunk_id not in existing_lookup:

        print("NEW:", chunk_id)

        new_chunks.append(chunk)
        new_ids.append(chunk_id)

    elif existing_lookup[chunk_id] != new_hash:

        print("UPDATED:", chunk_id)

        # Remove old version
        db.delete(ids=[chunk_id])

        # Add new version
        new_chunks.append(chunk)
        new_ids.append(chunk_id)

    else:

        print("UNCHANGED:", chunk_id)

# Insert all new/updated chunks
if new_chunks:
    db.add_documents(
        documents=new_chunks,
        ids=new_ids
    )

UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks0
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks1
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks2
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks3
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks4
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks5
UPDATED: Artificial_Intelligence_in_Healthcare.docx_chunks6
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks7
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks8
UNCHANGED: Artificial_Intelligence_in_Healthcare.docx_chunks9
UPDATED: Artificial_Intelligence_in_Healthcare.docx_chunks10
UPDATED: Artificial_Intelligence_in_Healthcare.docx_chunks11
NEW: Artificial_Intelligence_in_Healthcare.docx_chunks12
UPDATED: Cybersecurity_in_the_Digital_Age.docx_chunks13
UPDATED: Cybersecurity_in_the_Digital_Age.docx_chunks14
UPDATED: Cybersecurity_in_the_Digital_Age.docx_chunks15
UPDATED: Cybersecurity_in_the_Digit

In [68]:
result=db.similarity_search(
    "How Ai is use in healthcare and games",
    k=6
)

for doc in result:
    print("="*60)
    print(doc.metadata)
    print(doc.page_content)

{'source': 'docs\\Artificial_Intelligence_in_Healthcare.docx', 'hash': '212936a16e976b2325ddf0d1fa44c03c55a5a78834055447591b02befd1cc752'}
Artificial Intelligence in Healthcare

Informative Overview

Introduction

Artificial intelligence is increasingly becoming a foundational technology in modern healthcare. Rather than replacing doctors, nurses, researchers, and technicians, AI systems are primarily being developed to assist them with tasks involving large amounts of data, pattern recognition, prediction, automation, and decision support.
{'source': 'docs\\Artificial_Intelligence_in_Healthcare.docx', 'hash': '1363e93b0852c232cdeb4acd24ec6cd69f504f856a839d3f5e082d612579b752'}
Challenges and Future Direction

Healthcare AI carries serious responsibilities. Medical data is sensitive, so privacy and security must be considered throughout the system lifecycle. Models can reproduce biases present in historical datasets, perform poorly when deployed in environments different from their trai